In [ ]:
# config.py
import torch
import os
import matplotlib.pyplot as plt
from PIL import Image

# --- Training Hyperparameters ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE_GEN = 1e-4
LEARNING_RATE_DISC = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS_PRETRAIN = 50
NUM_EPOCHS_GAN = 100
HIGH_RES_SIZE = 96  # As per the paper
LOW_RES_SIZE = HIGH_RES_SIZE // 4
NUM_WORKERS = 4
LAMBDA_VGG = 1.0  # Weight for VGG/content loss
LAMBDA_ADV = 1e-3  # Weight for adversarial loss

os.makedirs("saved_models", exist_ok=True)

# --- Model Paths ---
PRETRAINED_GEN_PATH = "saved_models/srresnet_pretrained.pth"
GEN_PATH = "saved_models/generator.pth"
DISC_PATH = "saved_models/discriminator.pth"

# --- Dataset Paths ---
TRAIN_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_train_HR"
TEST_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_valid_HR"

# set the repo name
model_name = "keanteng/srgan-div2k-0723-v2"

In [ ]:
# dataset.py
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class ImageDataset(Dataset):
    """
    Custom dataset to load high-resolution images and create low-resolution counterparts.
    """
    def __init__(self, hr_dir, hr_size):
        super(ImageDataset, self).__init__()
        self.hr_image_files = [os.path.join(hr_dir, f) for f in os.listdir(hr_dir)]
        self.hr_size = hr_size

        # Transform for the original image before cropping
        self.initial_transform = transforms.Compose([
            transforms.ToTensor(),
        ])

        # Normalization transforms
        self.hr_normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalize to [-1, 1]
        self.lr_normalize = transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0]) # nothing change x = (x - mean) / std, so if mean=0 and std=1, x remains unchanged, we will use tanh to scale to [-1,1] in the generator

    def __getitem__(self, index):
        # Load image
        hr_image = Image.open(self.hr_image_files[index]).convert("RGB")

        # Convert to tensor first
        hr_tensor = self.initial_transform(hr_image)

        # Apply random crop to get consistent size
        crop_transform = transforms.RandomCrop(self.hr_size)
        hr_cropped = crop_transform(hr_tensor)

        # Create LR version by downsampling the cropped HR image
        lr_tensor = transforms.functional.resize(
            hr_cropped,
            size=self.hr_size // 4,
            interpolation=transforms.InterpolationMode.BICUBIC
        )

        # Apply normalization
        hr_normalized = self.hr_normalize(hr_cropped)
        lr_normalized = self.lr_normalize(lr_tensor)

        return lr_normalized, hr_normalized

    def __len__(self):
        return len(self.hr_image_files)

In [ ]:
from torch import nn

In [ ]:
class ResidualBlock(nn.Module):
    """
    A single residual block as defined in the SRGAN paper.
    It contains two convolutional layers with batch normalization and PReLU activation.
    """
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.PReLU(),
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(in_channels),
        )

    def forward(self, x):
        identity = x
        out = self.conv_block1(x)
        out = self.conv_block2(out)
        return identity + out

class UpsampleBlock(nn.Module):
    """
    Upsampling block using a convolutional layer and PixelShuffle.
    This increases the resolution by a factor of 2.
    """
    def __init__(self, in_channels, scale_factor=2):
        super(UpsampleBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * (scale_factor ** 2), kernel_size=3, stride=1, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(scale_factor)
        self.prelu = nn.PReLU()

    def forward(self, x):
        return self.prelu(self.pixel_shuffle(self.conv(x)))

In [ ]:
class Generator(nn.Module):
    """
    The Generator Network (SRResNet).
    It takes a low-resolution image and outputs a super-resolved version.
    """
    def __init__(self, in_channels=3, num_res_blocks=16):
        super(Generator, self).__init__()
        self.initial_conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=9, stride=1, padding=4),
            nn.PReLU()
        )

        self.residuals = nn.Sequential(*[ResidualBlock(64) for _ in range(num_res_blocks)])

        self.mid_conv = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64)
        )

        # Upsampling by 4x (two 2x upsample blocks)
        self.upsample_blocks = nn.Sequential(
            UpsampleBlock(64),
            UpsampleBlock(64),
        )

        self.final_conv = nn.Conv2d(64, in_channels, kernel_size=9, stride=1, padding=4)

    def forward(self, x):
        initial_out = self.initial_conv(x)
        residual_out = self.residuals(initial_out)
        mid_out = self.mid_conv(residual_out)
        mid_out = mid_out + initial_out # Skip connection
        upsampled_out = self.upsample_blocks(mid_out)
        final_out = self.final_conv(upsampled_out)
        return torch.tanh(final_out) # Tanh activation to scale output to [-1, 1]

In [ ]:
# evaluate.py
from torchvision.utils import save_image
from torchvision import transforms
import cv2
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import numpy as np
#import config
#from models import Generator

def calculate_psnr(img1, img2):
    """
    Calculate PSNR between two images.
    Images should be in range [0, 255] and of type uint8.
    """
    if img1.shape != img2.shape:
        raise ValueError("Input images must have the same dimensions")

    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:
        return float('inf')

    max_pixel = 255.0
    psnr_value = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr_value

def calculate_ssim(img1, img2):
    """
    Calculate SSIM between two images.
    Images should be in range [0, 255] and of type uint8.
    """
    if img1.shape != img2.shape:
        raise ValueError("Input images must have the same dimensions")

    # Convert to grayscale if images are color
    if len(img1.shape) == 3:
        img1_gray = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        img2_gray = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    else:
        img1_gray = img1
        img2_gray = img2

    ssim_value = ssim(img1_gray, img2_gray, data_range=255)
    return ssim_value

def tensor_to_numpy(tensor):
    """
    Convert tensor to numpy array in range [0, 255].
    """
    # Denormalize from [-1, 1] to [0, 1]
    tensor = tensor * 0.5 + 0.5
    # Clamp to [0, 1]
    tensor = torch.clamp(tensor, 0, 1)
    # Convert to numpy and scale to [0, 255]
    numpy_img = tensor.squeeze(0).cpu().detach().numpy()
    numpy_img = np.transpose(numpy_img, (1, 2, 0))  # CHW to HWC
    numpy_img = (numpy_img * 255).astype(np.uint8)
    return numpy_img

In [ ]:
from torchvision.utils import save_image
from torchvision import transforms

In [ ]:
# Load a test image
test_image_path = f"{TEST_DIR}/0801.png" # Example image
image = Image.open(test_image_path).convert("RGB")

# Prepare HR ground truth (crop to match output size)
hr_transform = transforms.Compose([
    transforms.Resize((HIGH_RES_SIZE, HIGH_RES_SIZE), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
# high res is 96 and 96 set by the paper we downscale using bicubic interpolation
# convert to tensor and scale to [-1, 1]
hr_image = hr_transform(image).unsqueeze(0).to(DEVICE)
# normalize to [-1, 1] for the generator input
# then remove the batch size
# then move to device

# Prepare LR image
lr_transform = transforms.Compose([
    transforms.Resize((LOW_RES_SIZE, LOW_RES_SIZE), interpolation=Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0]),
])
# low res is 4 times smaller than high res, so 24 and 24
# make it tensor and it will be range [0, 1]
# the normalize part did nothing
# x = (x - mean) / std, so if mean=0 and std=1, x remains unchanged
lr_image = lr_transform(image).unsqueeze(0).to(DEVICE)
# remove the batch size
# move to device

# Load generator
gen = Generator().to(DEVICE)
# load the generator class
gen.load_state_dict(torch.load(GEN_PATH, map_location=DEVICE))
# attach the weights to the generator
# the map location is to let the model has weights mapped on device
gen.eval()
# using eval mode to disable dropout and batch normalization (using running statistics)

with torch.no_grad():
    # disable gradient calculation for inference
    # we do not need to calculate gradients during inference
    # we do not want to update the weights
    sr_image = gen(lr_image)
    # here we get a generated images

# Save the results
os.makedirs("results", exist_ok=True)
save_image(sr_image * 0.5 + 0.5, "results/sr_result_1.png")
# we first make the image in the range of [0, 1] by multiplying by 0.5 and adding 0.5
# then save the image the function will scale the image to [0, 255] and save it as a PNG file (8bit image by default)
save_image(hr_image * 0.5 + 0.5, "results/hr_ground_truth_1.png")

# Create bicubic upscaled version for comparison
bicubic_image = lr_image.squeeze(0).cpu().detach()
# remove the batch size and move to CPU
# remove from gradient calculation because we are not doing any weight update
bicubic_image = (bicubic_image * 255).clamp(0, 255).byte()
# bicubic is lr at first and we make transforms.ToTensor() to make it 0 and 1 range
# then we make the range to be [0, 255] and convert to byte
# we clamp to make sure the values are in [0, 255] where value less than 0 will be 0 and greater than 255 will be 255
bicubic_transform = transforms.ToPILImage()
# a function to convert tensor to PIL image object
bicubic_pil = bicubic_transform(bicubic_image)
# now add the tensor in range [0,255] to the function
# the function does not help us to scale to 255 so we do ourselves
bicubic_pil = bicubic_pil.resize((HIGH_RES_SIZE, HIGH_RES_SIZE), Image.BICUBIC)
# now we resize the image to higher resolution using bicubic interpolation
bicubic_pil.save("results/bicubic_result_1.png")
# save the bicubic upscaled image

# Convert tensors to images for display
sr_display_img = transforms.ToPILImage()((sr_image.cpu().squeeze(0) * 0.5 + 0.5).clamp(0, 1))
hr_display_img = transforms.ToPILImage()((hr_image.cpu().squeeze(0) * 0.5 + 0.5).clamp(0, 1))
# eih, why here we don't multiply by 255
# we remove the batch size and move to CPU and scale to [0, 1] and clamp to make sure the values are in [0, 1]
# the function use the values inrange [0, 1] to convert to PIL image


# Display comparison
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title("Bicubic Input (Upscaled)")
plt.imshow(bicubic_pil)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("SRGAN Output")
plt.imshow(sr_display_img)
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Ground Truth")
plt.imshow(hr_display_img)
plt.axis('off')

plt.tight_layout()
plt.show()

# Convert images to numpy arrays for metric calculation
sr_numpy = tensor_to_numpy(sr_image)
hr_numpy = tensor_to_numpy(hr_image)
# convert to numpy array in range [0, 255]
bicubic_numpy = np.array(bicubic_pil)
# bicubic array is already in range [0, 255] as what we see above

# Calculate metrics
sr_psnr = calculate_psnr(hr_numpy, sr_numpy)
sr_ssim = calculate_ssim(hr_numpy, sr_numpy)
bicubic_psnr = calculate_psnr(hr_numpy, bicubic_numpy)
bicubic_ssim = calculate_ssim(hr_numpy, bicubic_numpy)

print("=== Evaluation Results ===")
print(f"SRGAN vs Ground Truth:")
print(f"  PSNR: {sr_psnr:.2f} dB")
print(f"  SSIM: {sr_ssim:.4f}")
print(f"\nBicubic vs Ground Truth:")
print(f"  PSNR: {bicubic_psnr:.2f} dB")
print(f"  SSIM: {bicubic_ssim:.4f}")
print(f"\nImprovement:")
print(f"  PSNR: +{sr_psnr - bicubic_psnr:.2f} dB")
print(f"  SSIM: +{sr_ssim - bicubic_ssim:.4f}")

print("\nEvaluation complete. Results saved in the 'results' folder.")

In [ ]:
# Create test dataset
test_dataset = ImageDataset(hr_dir=TEST_DIR, hr_size=HIGH_RES_SIZE)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=1)

# Load generator
gen = Generator().to(DEVICE)
gen.load_state_dict(torch.load(GEN_PATH, map_location=DEVICE))
gen.eval()

total_sr_psnr = 0
total_sr_ssim = 0
total_bicubic_psnr = 0
total_bicubic_ssim = 0
num_images = 0

print("Evaluating on test dataset...")

with torch.no_grad():
    for i, (lr, hr) in enumerate(test_loader):
        # the output from enumerate is a tuple of (index, (lr, hr))
        lr = lr.to(DEVICE)
        hr = hr.to(DEVICE)

        # Generate SR image
        sr = gen(lr)

        # Convert to numpy arrays
        sr_numpy = tensor_to_numpy(sr)
        hr_numpy = tensor_to_numpy(hr)

        # Create bicubic baseline
        lr_numpy = tensor_to_numpy(lr)
        # get the numpy array of the lr tensor
        bicubic_pil = Image.fromarray(lr_numpy).resize((HIGH_RES_SIZE, HIGH_RES_SIZE), Image.BICUBIC)
        # create an image object from the numpy array
        # adjust to the high resolution size using bicubic interpolation
        # this is a pil image object
        bicubic_numpy = np.array(bicubic_pil)
        # convert the image object to numpy array
        # here is goes back to the array

        # Calculate metrics
        sr_psnr = calculate_psnr(hr_numpy, sr_numpy)
        sr_ssim = calculate_ssim(hr_numpy, sr_numpy)
        bicubic_psnr = calculate_psnr(hr_numpy, bicubic_numpy)
        bicubic_ssim = calculate_ssim(hr_numpy, bicubic_numpy)
        # do the calculation using numpy arrays

        total_sr_psnr += sr_psnr
        total_sr_ssim += sr_ssim
        total_bicubic_psnr += bicubic_psnr
        total_bicubic_ssim += bicubic_ssim
        num_images += 1
        # update the totals and count
        # update count because we want to find average

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1} images...")
        # if 10 images are processed, print the progress

# Calculate averages
avg_sr_psnr = total_sr_psnr / num_images
avg_sr_ssim = total_sr_ssim / num_images
avg_bicubic_psnr = total_bicubic_psnr / num_images
avg_bicubic_ssim = total_bicubic_ssim / num_images
# find the average PSNR and SSIM for both SRGAN and bicubic

print(f"\n=== Average Results on {num_images} Test Images ===")
print(f"SRGAN:")
print(f"  Average PSNR: {avg_sr_psnr:.2f} dB")
print(f"  Average SSIM: {avg_sr_ssim:.4f}")
print(f"\nBicubic Baseline:")
print(f"  Average PSNR: {avg_bicubic_psnr:.2f} dB")
print(f"  Average SSIM: {avg_bicubic_ssim:.4f}")
print(f"\nAverage Improvement:")
print(f"  PSNR: +{avg_sr_psnr - avg_bicubic_psnr:.2f} dB")
print(f"  SSIM: +{avg_sr_ssim - avg_bicubic_ssim:.4f}")

In [ ]:
import pandas as pd
from pathlib import Path

def test_original_image(image_path, generator_model, save_prefix="test"):
    """
    Test SRGAN on a single original image and return the results.

    Args:
        image_path: Path to the original high-resolution image
        generator_model: The loaded generator model
        save_prefix: Prefix for saved images

    Returns:
        tuple: (lr_img, bicubic_img, sr_img, original_img) as PIL Images
    """
    # Load and prepare the original image
    original_img = Image.open(image_path).convert("RGB")

    # Get image dimensions and crop to make it divisible by 4
    width, height = original_img.size
    new_width = (width // 4) * 4 # interger division like 5 // 4 = 1 hen 1 * 4 = 4, also 8 // 4 = 2 then 2 * 4 = 8
    new_height = (height // 4) * 4
    original_img = original_img.crop((0, 0, new_width, new_height))
    # we should expect the crop is the same as the original here unless we use other data

    # Create low-resolution version (downscale by 4x)
    lr_size = (new_width // 4, new_height // 4)
    # now first compute a smaller size dimension
    lr_img = original_img.resize(lr_size, Image.BICUBIC)
    # downlsize the original image to low resolution
    # using bicubic interpolation

    # Create bicubic upscaled version
    bicubic_img = lr_img.resize((new_width, new_height), Image.BICUBIC)
    # for high resolution we use bicubic interpolation to upscale the low resolution image

    # Prepare LR image for the model
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0])
    ])
    # this function will scale LR to range [0,1]
    # basically ToTensor will do it the normalize will do nothing since mean and std are 0 and 1
    # we can remove it too if we want
    # just let it be a placeholder for future use

    lr_tensor = transform(lr_img).unsqueeze(0).to(DEVICE)
    # now make the lr to range [0,1]
    # thnen removethe batch size
    # then move to device

    # Generate SR image using the model
    with torch.no_grad():
        sr_tensor = generator_model(lr_tensor)

    # Convert SR tensor back to PIL Image
    sr_tensor = sr_tensor * 0.5 + 0.5  # Denormalize from [-1, 1] to [0, 1]
    sr_tensor = torch.clamp(sr_tensor, 0, 1)
    sr_numpy = sr_tensor.squeeze(0).cpu().numpy()
    sr_numpy = np.transpose(sr_numpy, (1, 2, 0))
    # from c, h, w to h, w, c
    sr_numpy = (sr_numpy * 255).astype(np.uint8)
    sr_img = Image.fromarray(sr_numpy)
    # convert the numpy array to PIL image object
    # can save as image

    # Create result folders
    os.makedirs("results", exist_ok = True)

    # Save the images if needed
    if save_prefix:
        lr_img.save(f"results/{save_prefix}_lr.png")
        bicubic_img.save(f"results/{save_prefix}_bicubic.png")
        sr_img.save(f"results/{save_prefix}_sr.png")
        original_img.save(f"results/{save_prefix}_original.png")

    return lr_img, bicubic_img, sr_img, original_img

def visualize_comparison_with_both_models(lr_img, bicubic_img, srresnet_img, srgan_img, original_img, title="Image Comparison"):
    """
    Display a side-by-side comparison of LR, Bicubic, SRResNet, SRGAN, and Original images.
    """
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))

    axes[0].imshow(lr_img)
    axes[0].set_title('Low Resolution')
    axes[0].axis('off')

    axes[1].imshow(bicubic_img)
    axes[1].set_title('Bicubic Upscaling')
    axes[1].axis('off')

    axes[2].imshow(srresnet_img)
    axes[2].set_title('SRResNet')
    axes[2].axis('off')

    axes[3].imshow(srgan_img)
    axes[3].set_title('SRGAN')
    axes[3].axis('off')

    axes[4].imshow(original_img)
    axes[4].set_title('Original HR')
    axes[4].axis('off')

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

def evaluate_both_models(image_paths, save_prefix="evaluation", show_images=True):
    """
    Evaluate both SRResNet and SRGAN on a list of specific image paths.

    Args:
        image_paths: List of image file paths to evaluate
        save_prefix: Prefix for saved result images
        show_images: Whether to display the image comparisons

    Returns:
        pandas.DataFrame: Table containing evaluation metrics for each image
    """
    # Load both models
    srresnet_gen = Generator().to(DEVICE)
    srresnet_gen.load_state_dict(torch.load(PRETRAINED_GEN_PATH, map_location=DEVICE))
    srresnet_gen.eval()

    srgan_gen = Generator().to(DEVICE)
    srgan_gen.load_state_dict(torch.load(GEN_PATH, map_location=DEVICE))
    srgan_gen.eval()

    results = []

    print(f"Evaluating {len(image_paths)} specific images with both models...")

    for i, image_path in enumerate(image_paths):
        print(f"Processing image {i+1}/{len(image_paths)}: {Path(image_path).name}")

        try:
            # Test with SRResNet
            lr_img, bicubic_img, srresnet_img, original_img = test_original_image(
                image_path, srresnet_gen, f"{save_prefix}_srresnet_{i+1}"
            )

            # Test with SRGAN
            _, _, srgan_img, _ = test_original_image(
                image_path, srgan_gen, f"{save_prefix}_srgan_{i+1}"
            )
            # we put dash because we don't want the other samples
            # bicubic, hd and so on we have from SRResNet already

            # Show comparison if requested
            if show_images:
                # set to true by default
                visualize_comparison_with_both_models(
                    lr_img, bicubic_img, srresnet_img, srgan_img, original_img,
                    title=f"Comparison for {Path(image_path).name}"
                )

            # Convert to numpy arrays for metric calculation
            # conver the images to numpy arrays
            srresnet_numpy = np.array(srresnet_img)
            srgan_numpy = np.array(srgan_img)
            original_numpy = np.array(original_img)
            bicubic_numpy = np.array(bicubic_img)

            # Calculate metrics for SRResNet
            srresnet_psnr = calculate_psnr(original_numpy, srresnet_numpy)
            srresnet_ssim = calculate_ssim(original_numpy, srresnet_numpy)

            # Calculate metrics for SRGAN
            srgan_psnr = calculate_psnr(original_numpy, srgan_numpy)
            srgan_ssim = calculate_ssim(original_numpy, srgan_numpy)

            # Calculate metrics for Bicubic
            bicubic_psnr = calculate_psnr(original_numpy, bicubic_numpy)
            bicubic_ssim = calculate_ssim(original_numpy, bicubic_numpy)

            # Store results
            result = {
                'Image': Path(image_path).name,
                'SRResNet_PSNR': srresnet_psnr,
                'SRResNet_SSIM': srresnet_ssim,
                'SRGAN_PSNR': srgan_psnr,
                'SRGAN_SSIM': srgan_ssim,
                'Bicubic_PSNR': bicubic_psnr,
                'Bicubic_SSIM': bicubic_ssim,
                'SRResNet_PSNR_Improvement': srresnet_psnr - bicubic_psnr,
                'SRResNet_SSIM_Improvement': srresnet_ssim - bicubic_ssim,
                'SRGAN_PSNR_Improvement': srgan_psnr - bicubic_psnr,
                'SRGAN_SSIM_Improvement': srgan_ssim - bicubic_ssim
            }
            results.append(result)
            # every loop will append the result for each image

        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            # Add error entry to maintain table structure
            result = {
                'Image': Path(image_path).name,
                'SRResNet_PSNR': np.nan,
                'SRResNet_SSIM': np.nan,
                'SRGAN_PSNR': np.nan,
                'SRGAN_SSIM': np.nan,
                'Bicubic_PSNR': np.nan,
                'Bicubic_SSIM': np.nan,
                'SRResNet_PSNR_Improvement': np.nan,
                'SRResNet_SSIM_Improvement': np.nan,
                'SRGAN_PSNR_Improvement': np.nan,
                'SRGAN_SSIM_Improvement': np.nan
            }
            results.append(result)
            # if we face error add NaN to the row because the for loop inside we use try and except

    # Create DataFrame
    df = pd.DataFrame(results)

    # Calculate summary statistics (excluding NaN values)
    summary = {
        'Image': 'AVERAGE',
        'SRResNet_PSNR': df['SRResNet_PSNR'].mean(),
        'SRResNet_SSIM': df['SRResNet_SSIM'].mean(),
        'SRGAN_PSNR': df['SRGAN_PSNR'].mean(),
        'SRGAN_SSIM': df['SRGAN_SSIM'].mean(),
        'Bicubic_PSNR': df['Bicubic_PSNR'].mean(),
        'Bicubic_SSIM': df['Bicubic_SSIM'].mean(),
        'SRResNet_PSNR_Improvement': df['SRResNet_PSNR_Improvement'].mean(),
        'SRResNet_SSIM_Improvement': df['SRResNet_SSIM_Improvement'].mean(),
        'SRGAN_PSNR_Improvement': df['SRGAN_PSNR_Improvement'].mean(),
        'SRGAN_SSIM_Improvement': df['SRGAN_SSIM_Improvement'].mean()
    }
    # furhter summarize the dataframe

    # Add summary row
    df = pd.concat([df, pd.DataFrame([summary])], ignore_index=True)
    # so we have a summary table

    return df

def display_results_table(df, title="Model Comparison Results"):
    """
    Display the results table with nice formatting
    """
    print(f"\n{'='*120}") # print 120 characters of equal sign
    print(f"{title:^120}") # print a centered within a field of 120 characters the ^ is center
    print(f"{'='*120}")

    # Format the DataFrame for display
    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)

    print(df.to_string(index=False))
    # print the dataframe without index
    # print it to the console
    print(f"{'='*120}\n")